# PHASE 1 : Analyse Exploratoire (EDA) et Nettoyage des Données

In [8]:
import pandas as pd


In [12]:
df = pd.read_csv("data/dataset.csv", low_memory=False)

# distribution des classes
print(df[' Label'].value_counts())

# valeurs manquantes
print(df.isnull().sum().sum())


import numpy as np
print((df == np.inf).sum().sum())
print((df == -np.inf).sum().sum())

 Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64
1358
4376
0


In [14]:
df.shape

(2830743, 79)

In [16]:

# nettoyage des noms de colonnes 
df.columns = df.columns.str.strip()
print("Colonnes nettoyées")



Colonnes nettoyées


In [18]:
# remplacement de inf par NaN puis drop 
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
print(f"Suppression NaN/Inf : {df.shape}")



Suppression NaN/Inf : (2827876, 79)


In [20]:
# suppression des doublons
df.drop_duplicates(inplace=True)
print(f"suppression doublons : {df.shape}")



suppression doublons : (2520798, 79)


In [21]:
# normalisation des labels
df['Label'] = df['Label'].str.strip()
df['Label'] = df['Label'].replace({
    'Web Attack \x96 Brute Force' : 'Web Attack - Brute Force',
    'Web Attack \x96 XSS'         : 'Web Attack - XSS',
    'Web Attack \x96 Sql Injection': 'Web Attack - Sql Injection'
})


In [24]:
# encodage en binaire 
df['label_binary'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

print(f"\n Distribution finale :")
print(df['Label'].value_counts())
print(f"\n Anomalies : {df['label_binary'].sum():,}")
print(f"\n Normal    : {(df['label_binary'] == 0).sum():,}")


 Distribution finale :
Label
BENIGN                        2095057
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

 Anomalies : 425,741

 Normal    : 2,095,057


In [26]:

df['Label'] = df['Label'].replace({
    'Web Attack \ufffd Brute Force' : 'Web Attack - Brute Force',
    'Web Attack \ufffd XSS'         : 'Web Attack - XSS',
    'Web Attack \ufffd Sql Injection': 'Web Attack - Sql Injection'
})

print(df['Label'].unique())

['BENIGN' 'DDoS' 'PortScan' 'Bot' 'Infiltration'
 'Web Attack - Brute Force' 'Web Attack - XSS'
 'Web Attack - Sql Injection' 'FTP-Patator' 'SSH-Patator' 'DoS slowloris'
 'DoS Slowhttptest' 'DoS Hulk' 'DoS GoldenEye' 'Heartbleed']


# PHASE 2 : Feature Engineering

In [29]:
from sklearn.ensemble import RandomForestClassifier

X = df.drop(columns=['Label', 'label_binary'])
y = df['label_binary']

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns)
top20 = importances.nlargest(20)
print(top20)

Avg Bwd Segment Size           0.082174
Packet Length Std              0.077700
Packet Length Variance         0.068059
Max Packet Length              0.063001
Bwd Packet Length Std          0.051223
Bwd Packet Length Max          0.050450
Average Packet Size            0.042318
Bwd Packet Length Mean         0.034722
Total Length of Bwd Packets    0.032192
Packet Length Mean             0.026453
Destination Port               0.025147
Subflow Fwd Bytes              0.021930
Subflow Fwd Packets            0.020510
Total Fwd Packets              0.020280
PSH Flag Count                 0.019343
act_data_pkt_fwd               0.018339
Total Length of Fwd Packets    0.017380
Subflow Bwd Bytes              0.017296
Fwd Header Length.1            0.015679
Fwd Header Length              0.014658
dtype: float64


In [31]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# top 20
top20_features = top20.index.tolist()
X_selected = df[top20_features].copy()
y = df['label_binary'].copy()

# normalisation
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_selected)
X_scaled = pd.DataFrame(X_scaled, columns=top20_features)

print(f" Shape final : {X_scaled.shape}")
print(f" Anomalies  : {y.sum():,}")
print(f" Normal     : {(y==0).sum():,}")

# sauvegarder scaler pour plus tard
import joblib
joblib.dump(scaler, 'scaler.pkl')
print(" Scaler sauvegardé")

 Shape final : (2520798, 20)
 Anomalies  : 425,741
 Normal     : 2,095,057
 Scaler sauvegardé


In [33]:


# dataset final (features sélectionnées + normalisées)
X_scaled.to_csv("X_scaled.csv", index=False)

# labels
y.to_csv("y.csv", index=False)

# top 20 features
joblib.dump(top20_features, 'top20_features.pkl')

print("Tout sauvegardé")

Tout sauvegardé
